# RMT-PPAD migration NB81 - Phase P2 (GCA -> CLRHead channel adapter)

**Purpose.** Verify `GCAtoCLRAdapter` projects a 3-level 256-channel
pyramid into a 3-level 64-channel pyramid with spatial shape preserved.
Implemented per `appendix-path3-implementation-prompt.md` sec 5.2.

**Acceptance criterion (sec 5.3):** shape unit test:
```
[(2,256,80,80), (2,256,40,40), (2,256,20,20)]
    -> [(2,64,80,80), (2,64,40,40), (2,64,20,20)]
```

Plus our additions: backward-pass gradient check, wrong-input-length
raises ValueError.

**Wall time:** a few seconds. No Drive I/O, no dataset, just a CPU
module instantiation + forward/backward pass.

### Cell 1: Mount Drive, set REPO_ROOT, sanity-check torch

In [1]:
import os, sys
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing {REPO_ROOT} -- verify Drive sync.')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
print('torch', torch.__version__)
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
torch 2.10.0+cpu
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


### Cell 2: Run the smoke test (appendix acceptance)
Uses `run_streaming` so the subprocess output reaches the cell directly.

In [2]:
import sys, os
ADAPTER_SCRIPT = 'stage2/rmt_ppad_migration/P2_adapter/tools/gca_to_clr_adapter.py'
log = os.path.join(LOG_DIR, 'NB81_adapter_smoke.log')
rc = run_streaming([sys.executable, '-u', ADAPTER_SCRIPT], log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'GCAtoCLRAdapter smoke test failed with rc={rc}; see {log}')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P2_adapter/tools/gca_to_clr_adapter.py
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB81_adapter_smoke.log
[smoke] GCAtoCLRAdapter starting
[smoke] params = 49,536
[smoke] level 0: in (2, 256, 80, 80) -> out (2, 64, 80, 80)  (expect (2, 64, 80, 80))
[smoke] level 1: in (2, 256, 40, 40) -> out (2, 64, 40, 40)  (expect (2, 64, 40, 40))
[smoke] level 2: in (2, 256, 20, 20) -> out (2, 64, 20, 20)  (expect (2, 64, 20, 20))
[smoke] backward pass: all 3 projections received non-zero gradient
[smoke] wrong-length input correctly raised: GCAtoCLRAdapter expected 3 pyramid levels, got 2.
[smoke] PASS
[run_streaming] return_code=0


### Cell 3: Import the adapter in-process and dump its module tree
Confirms the class can be imported into a running notebook (P4 will need
to do exactly this), and prints the named modules / param count for the
record.

In [3]:
import sys
TOOLS = 'stage2/rmt_ppad_migration/P2_adapter/tools'
if TOOLS not in sys.path:
    sys.path.insert(0, TOOLS)
from gca_to_clr_adapter import GCAtoCLRAdapter

adapter = GCAtoCLRAdapter(in_channels=256, out_channels=64)
n_params = sum(p.numel() for p in adapter.parameters())
print(f'GCAtoCLRAdapter total params = {n_params:,}')
print()
print('Named modules:')
for name, m in adapter.named_modules():
    print(f'  {name or "<root>":40s} {type(m).__name__}')

# One more sanity: forward on the exact appendix shapes.
import torch
with torch.no_grad():
    out = adapter([torch.randn(2, 256, 80, 80),
                   torch.randn(2, 256, 40, 40),
                   torch.randn(2, 256, 20, 20)])
print('\nIn-process forward:', [tuple(t.shape) for t in out])
expected = [(2, 64, 80, 80), (2, 64, 40, 40), (2, 64, 20, 20)]
assert [tuple(t.shape) for t in out] == expected, 'shape mismatch'
print('\n[P2 result] PASS')

GCAtoCLRAdapter total params = 49,536

Named modules:
  <root>                                   GCAtoCLRAdapter
  proj                                     ModuleList
  proj.0                                   Sequential
  proj.0.0                                 Conv2d
  proj.0.1                                 BatchNorm2d
  proj.0.2                                 ReLU
  proj.1                                   Sequential
  proj.1.0                                 Conv2d
  proj.1.1                                 BatchNorm2d
  proj.1.2                                 ReLU
  proj.2                                   Sequential
  proj.2.0                                 Conv2d
  proj.2.1                                 BatchNorm2d
  proj.2.2                                 ReLU

In-process forward: [(2, 64, 80, 80), (2, 64, 40, 40), (2, 64, 20, 20)]

[P2 result] PASS
